# CS781 Legal NER — Clean Experiment Notebook

**Purpose:** one clean, reproducible notebook for checkpoint/data inventory, a locked exact-span evaluation harness, RESP/WIT diagnostics, decoding-bias diagnostics, and safe candidate comparisons.

## Safety and experiment policy
- Uses the best established NER checkpoint (`champion_recovery.pt`) as the champion and the Phase-1 checkpoint as its encoder source.
- Searches all attached Kaggle datasets for backup datasets/checkpoints and prints what is present.
- Other checkpoints are inventoried and compatibility-checked; a filename containing `latest` does **not** automatically make it the best NER checkpoint.
- No training occurs in this notebook. Earlier short fine-tuning and frozen-probe experiments regressed, so training is deferred until the diagnostics identify a specific hypothesis.
- Never overwrites `prediction_champion_plus_rules.json`, any checkpoint, or any existing prediction file.
- Fold 1 is the training/development fold; fold 2 is a repeatedly used validation/development fold; fold 3 is test and must not be used for tuning.
- Exact-span metrics are reported for the prediction pipeline actually run. The threshold-only baseline is not falsely presented as the champion-plus-rules score of 0.299526.

In [ ]:
# 1. Necessary imports only + runtime
import os, json, re, gc, math, copy
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
from tokenizers import Tokenizer
from transformers import GPT2Config, GPT2Model
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Inventory all available datasets/checkpoints before choosing anything
ROOTS = [Path("/kaggle/input"), Path("/kaggle/working")]
CHECKPOINT_NAMES = {
    "champion_recovery.pt", "best_ner_model.pt", "checkpoint_latest.pt",
    "targeted_pilot_8_steps.pt", "best_model.pt", "model.pt"
}
DATASET_NAMES = {
    "legal_bpe_32k.json", "fold1-text.json", "fold1-gold.json",
    "fold2-text.json", "fold2-gold.json", "fold3-text.json",
    "train.json", "test.json"
}
all_files = []
for root in ROOTS:
    if root.exists():
        all_files.extend(p for p in root.rglob("*") if p.is_file())

all_files = sorted(set(all_files))
checkpoint_files = [p for p in all_files if p.name in CHECKPOINT_NAMES or p.suffix.lower() in {".pt", ".pth"}]
dataset_files = [p for p in all_files if p.name in DATASET_NAMES or ("fold" in p.name.lower() and p.suffix.lower() == ".json")]

print("=== CHECKPOINT INVENTORY ===")
if checkpoint_files:
    for p in checkpoint_files:
        print(f"{p}  [{p.stat().st_size / 1024**2:.1f} MiB]")
else:
    print("No checkpoint files found.")

print("\n=== DATASET/TOKENIZER INVENTORY ===")
for p in dataset_files:
    print(p)

print("\n=== BACKUP DATASET DIRECTORIES ===")
for p in sorted({str(x.parent) for x in dataset_files}):
    print(p)

# Pick the recovery champion explicitly; don't accidentally choose an arbitrary checkpoint_latest.pt.
champion_candidates = [p for p in checkpoint_files if p.name == "champion_recovery.pt"]
if not champion_candidates:
    champion_candidates = [p for p in checkpoint_files if p.name == "best_ner_model.pt"]
if not champion_candidates:
    raise FileNotFoundError(
        "No champion_recovery.pt or best_ner_model.pt found under /kaggle/input or /kaggle/working. "
        "Attach the CS781 recovery/checkpoint dataset in Kaggle."
    )

def rank_champion_path(p):
    s = str(p).lower()
    return (
        0 if "cs781_recovery" in s else 1,
        0 if "working-backup" in s else 1,
        0 if "/kaggle/input/" in s else 1,
        s
    )

CHAMPION_PATH = sorted(champion_candidates, key=rank_champion_path)[0]

phase1_candidates = [
    p for p in checkpoint_files
    if p.name == "checkpoint_latest.pt" and "cs781-phase-1-checkpoints" in str(p)
]
if not phase1_candidates:
    phase1_candidates = [
        p for p in checkpoint_files
        if p.name == "checkpoint_latest.pt" and "phase-1" in str(p).lower()
    ]
if not phase1_candidates:
    phase1_candidates = [p for p in checkpoint_files if p.name == "checkpoint_latest.pt"]

if not phase1_candidates:
    raise FileNotFoundError("No Phase-1 checkpoint_latest.pt found.")

PHASE1_PATH = sorted(phase1_candidates, key=lambda p: (0 if "/kaggle/input/" in str(p) else 1, str(p)))[0]

print("\nSELECTED NER CHAMPION:", CHAMPION_PATH)
print("SELECTED PHASE-1 ENCODER:", PHASE1_PATH)
print("Selection policy: recovery champion for NER; Phase-1 checkpoint for encoder.")

In [ ]:
# 3. Inspect checkpoint metadata; only the known champion is selected for NER inference
def safe_torch_load(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")

recovery = safe_torch_load(CHAMPION_PATH)
phase1_ckpt = safe_torch_load(PHASE1_PATH)

print("Champion checkpoint keys:", list(recovery.keys()) if isinstance(recovery, dict) else type(recovery))
print("Phase-1 checkpoint keys:", list(phase1_ckpt.keys()) if isinstance(phase1_ckpt, dict) else type(phase1_ckpt))
if isinstance(phase1_ckpt, dict):
    print("Phase-1 step/epoch metadata:", {
        k: phase1_ckpt.get(k) for k in ("step", "global_step", "epoch", "best_step")
        if k in phase1_ckpt
    })

# Report other checkpoints' metadata without silently replacing the champion.
print("\n=== OTHER CHECKPOINT METADATA ===")
for p in checkpoint_files:
    if p in (CHAMPION_PATH, PHASE1_PATH):
        continue
    try:
        obj = safe_torch_load(p)
        if isinstance(obj, dict):
            keys = list(obj.keys())
            meta = {k: obj.get(k) for k in ("step", "global_step", "epoch", "optimizer_steps")
                    if k in obj and isinstance(obj.get(k), (int, float, str))}
            state = obj.get("model_state_dict")
            shape_info = None
            if isinstance(state, dict):
                cls = state.get("classifier.weight")
                if cls is not None and hasattr(cls, "shape"):
                    shape_info = tuple(cls.shape)
            print(f"{p}: keys={keys[:12]}, metadata={meta}, classifier.weight={shape_info}")
        else:
            print(f"{p}: object={type(obj).__name__}")
        del obj
    except Exception as e:
        print(f"{p}: could not inspect ({type(e).__name__}: {e})")

In [ ]:
# 4. Reconstruct model and load tokenizer/data files
BIO_LABELS = list(recovery["bio_labels"])
label2id = {label: i for i, label in enumerate(BIO_LABELS)}
id2label = {i: label for i, label in enumerate(BIO_LABELS)}
ENTITY_LABELS = [
    "APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT",
    "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"
]
assert len(BIO_LABELS) == 25, f"Expected 25 BIO labels, got {len(BIO_LABELS)}"

cfg = phase1_ckpt.get("model_config")
if cfg is None:
    raise KeyError("Phase-1 checkpoint has no model_config; cannot reconstruct encoder safely.")

gpt2_config = GPT2Config(
    vocab_size=cfg["vocab_size"],
    n_positions=cfg["context_length"],
    n_ctx=cfg["context_length"],
    n_embd=cfg["hidden_size"],
    n_layer=cfg["num_layers"],
    n_head=cfg["num_heads"],
    resid_pdrop=0.0, embd_pdrop=0.0, attn_pdrop=0.0
)

class LegalNERModel(nn.Module):
    def __init__(self, pretrained_lm, num_labels):
        super().__init__()
        self.encoder = pretrained_lm.transformer
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(pretrained_lm.config.hidden_size, num_labels)
    def forward(self, input_ids, attention_mask=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return self.classifier(self.dropout(outputs.last_hidden_state))

phase1_lm = GPT2Model(gpt2_config)
phase1_state = {
    k.removeprefix("transformer."): v
    for k, v in phase1_ckpt["model_state_dict"].items()
    if k.startswith("transformer.")
}
phase1_lm.load_state_dict(phase1_state, strict=True)
lm_wrapper = type("LMWrapper", (), {"transformer": phase1_lm, "config": gpt2_config})()
champion_model = LegalNERModel(lm_wrapper, len(BIO_LABELS)).to(device)
champion_model.load_state_dict(recovery["model_state_dict"], strict=True)
champion_model.eval()

def find_named_file(filename, preferred_tokens=()):
    matches = [p for p in all_files if p.name == filename]
    if not matches:
        raise FileNotFoundError(f"Required file {filename} not found in Kaggle input/working.")
    preferred = [p for p in matches if all(t.lower() in str(p).lower() for t in preferred_tokens)]
    candidates = preferred or matches
    return sorted(candidates, key=lambda p: (0 if "/kaggle/input/" in str(p) else 1, str(p)))[0]

tokenizer_path = find_named_file("legal_bpe_32k.json")
fold1_text_path = find_named_file("fold1-text.json")
fold1_gold_path = find_named_file("fold1-gold.json")
fold2_text_path = find_named_file("fold2-text.json")
fold2_gold_path = find_named_file("fold2-gold.json")
fold3_text_path = find_named_file("fold3-text.json")

tokenizer = Tokenizer.from_file(str(tokenizer_path))
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train_texts = load_json(fold1_text_path)
train_gold = load_json(fold1_gold_path)
val_texts = load_json(fold2_text_path)
val_gold = load_json(fold2_gold_path)
test_texts = load_json(fold3_text_path)

# Normalize document IDs to strings consistently.
train_texts = {str(k): v for k, v in train_texts.items()}
train_gold = {str(k): v for k, v in train_gold.items()}
val_texts = {str(k): v for k, v in val_texts.items()}
val_gold = {str(k): v for k, v in val_gold.items()}
test_texts = {str(k): v for k, v in test_texts.items()}

print("Champion loaded:", CHAMPION_PATH)
print("Tokenizer:", tokenizer_path, "| vocab size:", tokenizer.get_vocab_size())
print("Documents: train", len(train_texts), "| validation", len(val_texts), "| test", len(test_texts))
print("Train/val IDs disjoint:", set(train_texts).isdisjoint(val_texts))
print("Validation text/gold IDs match:", set(val_texts) == set(val_gold))
print("Train text/gold IDs match:", set(train_texts) == set(train_gold))
assert set(train_texts) == set(train_gold)
assert set(val_texts) == set(val_gold)

In [ ]:
# 5. Find and parse champion thresholds
threshold_candidates = [
    p for p in all_files
    if p.suffix.lower() == ".json" and "threshold" in p.name.lower()
]
print("Threshold files found:")
for p in threshold_candidates:
    print(" ", p)

def extract_thresholds(obj):
    if not isinstance(obj, dict):
        return None
    for key in ("champ_thr", "thresholds", "label_thresholds", "threshold_by_label"):
        if isinstance(obj.get(key), dict) and obj[key] and all(isinstance(v, (int, float)) for v in obj[key].values()):
            return {str(k): float(v) for k, v in obj[key].items()}
    if obj and all(isinstance(v, (int, float)) for v in obj.values()):
        return {str(k): float(v) for k, v in obj.items()}
    return None

champ_thr = None
threshold_path_used = None
for p in sorted(threshold_candidates, key=lambda p: (
    0 if "champion_thresholds_recovery" in p.name.lower() else 1,
    0 if "cs781_recovery" in str(p).lower() else 1,
    str(p)
)):
    try:
        obj = load_json(p)
        found = extract_thresholds(obj)
        if found and all(lab in found for lab in ENTITY_LABELS):
            champ_thr, threshold_path_used = found, p
            break
    except Exception:
        pass

if champ_thr is None:
    # These are the documented recovery thresholds from the existing notebook.
    champ_thr = {
        "APP": 0.15, "RESP": 0.45, "A.COUNSEL": 0.45, "R.COUNSEL": 0.45,
        "JUDGE": 0.25, "WIT": 0.45, "AUTH": 0.35, "COURT": 0.45,
        "STAT": 0.25, "PREC": 0.35, "DATE": 0.55, "CASENO": 0.35
    }
    print("WARNING: no usable threshold JSON found; using documented recovery thresholds.")
else:
    print("Loaded thresholds from:", threshold_path_used)
print("Thresholds:", champ_thr)

In [ ]:
# 6. Core inference, BIO decoding, and exact-span evaluation helpers
from bisect import bisect_left, bisect_right

NUM_LABELS = len(BIO_LABELS)
legal_start = np.array([tag == "O" or tag.startswith("B-") for tag in BIO_LABELS], dtype=bool)
legal_transition = np.ones((NUM_LABELS, NUM_LABELS), dtype=bool)
for prev_id, prev_tag in enumerate(BIO_LABELS):
    for curr_id, curr_tag in enumerate(BIO_LABELS):
        if curr_tag.startswith("I-"):
            legal_transition[prev_id, curr_id] = prev_tag in (f"B-{curr_tag[2:]}", f"I-{curr_tag[2:]}")

def bio_viterbi_decode(logits):
    emissions = logits.detach().float().cpu().numpy() if isinstance(logits, torch.Tensor) else np.asarray(logits, dtype=np.float64)
    if emissions.ndim != 2 or emissions.shape[1] != NUM_LABELS:
        raise ValueError(f"Expected [sequence_length, {NUM_LABELS}] emissions, got {emissions.shape}")
    if len(emissions) == 0:
        return []
    emissions = emissions.astype(np.float64, copy=False)
    neg_inf = -1e12
    scores = np.full_like(emissions, neg_inf, dtype=np.float64)
    back = np.zeros(emissions.shape, dtype=np.int32)
    scores[0, legal_start] = emissions[0, legal_start]
    for t in range(1, len(emissions)):
        cand = np.where(legal_transition, scores[t-1][:, None], neg_inf) + emissions[t][None, :]
        back[t] = np.argmax(cand, axis=0)
        scores[t] = cand[back[t], np.arange(NUM_LABELS)]
    last = int(np.argmax(scores[-1]))
    path = [last]
    for t in range(len(emissions)-1, 0, -1):
        path.append(int(back[t, path[-1]]))
    return list(reversed(path))

def decode_spans(text, offsets, pred_ids):
    spans, current_label, current_start, current_end = [], None, None, None
    def close():
        nonlocal current_label, current_start, current_end
        if current_label is not None:
            s, e = current_start, current_end
            while s < e and text[s].isspace(): s += 1
            while e > s and text[e-1].isspace(): e -= 1
            if s < e: spans.append((s, e, current_label))
        current_label = current_start = current_end = None
    for (s, e), pred_id in zip(offsets, pred_ids):
        if e <= s:
            continue
        tag = BIO_LABELS[int(pred_id)]
        if tag == "O":
            close(); continue
        prefix, lab = tag.split("-", 1)
        if prefix == "B" or current_label != lab:
            close(); current_label, current_start, current_end = lab, s, e
        else:
            current_end = e
    close()
    return sorted(set(spans))

DIAG_MAX_LEN, DIAG_STRIDE = 512, 448
@torch.inference_mode()
def diag_doc_probs(model, text):
    enc = tokenizer.encode(text, add_special_tokens=False)
    ids, offsets = enc.ids, enc.offsets
    n = len(ids)
    if not n:
        return offsets, torch.zeros((0, len(BIO_LABELS)))
    logit_sum = torch.zeros((n, len(BIO_LABELS)), dtype=torch.float32)
    counts = torch.zeros(n, dtype=torch.float32)
    model.eval()
    for start in range(0, n, DIAG_STRIDE):
        end = min(start + DIAG_MAX_LEN, n)
        x = torch.tensor([ids[start:end]], dtype=torch.long, device=device)
        logits = model(x, torch.ones_like(x))[0].float().cpu()
        logit_sum[start:end] += logits
        counts[start:end] += 1
        if end == n: break
    avg_logits = logit_sum / counts.clamp_min(1).unsqueeze(-1)
    return offsets, torch.softmax(avg_logits, dim=-1)

def predict_texts(model, texts):
    result = {}
    model.eval()
    for doc_id, text in tqdm(texts.items(), desc="Predicting", leave=False):
        offsets, probs = diag_doc_probs(model, text)
        if not len(offsets):
            result[doc_id] = []
            continue
        pred_ids = torch.tensor(bio_viterbi_decode(torch.log(probs.clamp_min(1e-12))), dtype=torch.long)
        starts, ends = [a for a, b in offsets], [b for a, b in offsets]
        spans = []
        for s, e, lab in decode_spans(text, offsets, pred_ids.tolist()):
            ix = [i for i in range(bisect_right(ends, s), bisect_left(starts, e)) if ends[i] > starts[i]]
            conf = float(probs[ix, pred_ids[ix]].mean()) if ix else 0.0
            spans.append((int(s), int(e), lab, conf))
        result[doc_id] = spans
    return result

def apply_thr(raw, thresholds):
    return {d: [(s, e, lab) for s, e, lab, conf in spans if conf >= thresholds.get(lab, 0.0)] for d, spans in raw.items()}

def calculate_exact_span_metrics(gold_by_doc, pred_by_doc, labels=ENTITY_LABELS):
    metrics = {}
    for lab in labels:
        gold_set, pred_set = set(), set()
        for d, ents in gold_by_doc.items():
            for ent in ents:
                if len(ent) >= 3 and str(ent[2]) == lab:
                    gold_set.add((str(d), int(ent[0]), int(ent[1]), lab))
        for d, spans in pred_by_doc.items():
            for ent in spans:
                if len(ent) >= 3 and str(ent[2]) == lab:
                    pred_set.add((str(d), int(ent[0]), int(ent[1]), lab))
        tp = len(gold_set & pred_set); fp = len(pred_set - gold_set); fn = len(gold_set - pred_set)
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        f = 2*p*r/(p+r) if p+r else 0.0
        metrics[lab] = {"gold":len(gold_set), "pred":len(pred_set), "TP":tp, "FP":fp, "FN":fn,
                        "precision":p, "recall":r, "f1":f}
    metrics["MACRO_F1"] = sum(metrics[x]["f1"] for x in labels) / len(labels)
    return metrics

# Verify exact recovery weights and a forward pass.
state = recovery["model_state_dict"]
loaded = champion_model.state_dict()
assert set(state) == set(loaded)
max_diff = max((loaded[k].detach().cpu() - state[k]).abs().max().item() for k in state)
assert max_diff == 0.0
with torch.inference_mode():
    z = champion_model(torch.zeros((1, 8), dtype=torch.long, device=device), torch.ones((1, 8), dtype=torch.long, device=device))
assert tuple(z.shape) == (1, 8, len(BIO_LABELS))
print("Champion weights verified exactly; output shape:", tuple(z.shape))

In [ ]:
# 7. Baseline: raw champion and thresholded champion, using the SAME evaluator
raw_val = predict_texts(champion_model, val_texts)
threshold_val = apply_thr(raw_val, champ_thr)
raw_metrics = calculate_exact_span_metrics(val_gold, {
    d: [(s,e,l) for s,e,l,c in spans] for d, spans in raw_val.items()
})
threshold_metrics = calculate_exact_span_metrics(val_gold, threshold_val)

def print_metrics(title, m):
    print("\n" + title, "| macro-F1 =", round(m["MACRO_F1"], 6))
    print(f"{'LABEL':12} {'P':>7} {'R':>7} {'F1':>7} {'TP':>5} {'FP':>5} {'FN':>5} {'PRED':>6}")
    for lab in ENTITY_LABELS:
        x = m[lab]
        print(f"{lab:12} {x['precision']:7.3f} {x['recall']:7.3f} {x['f1']:7.3f} {x['TP']:5d} {x['FP']:5d} {x['FN']:5d} {x['pred']:6d}")

print_metrics("RAW CHAMPION", raw_metrics)
print_metrics("THRESHOLD-ONLY CHAMPION", threshold_metrics)
print("\nReference from prior established champion+rules pipeline: 0.299526")
print("Important: threshold-only score is NOT expected to equal the champion+rules reference.")

In [ ]:
# 8. Annotation/data audit: counts, document coverage, concentration, role-context examples
def label_audit(texts, gold, labels=("RESP", "WIT")):
    for lab in labels:
        doc_counts, surface_counts = Counter(), Counter()
        examples = []
        for doc_id, ents in gold.items():
            for ent in ents:
                if len(ent) < 3 or str(ent[2]) != lab: continue
                s, e = int(ent[0]), int(ent[1])
                doc_counts[str(doc_id)] += 1
                surface_counts[texts[str(doc_id)][s:e].strip()] += 1
                if len(examples) < 8:
                    text = texts[str(doc_id)]
                    examples.append((str(doc_id), s, e, text[max(0,s-100):s], text[s:e], text[e:min(len(text),e+120)]))
        total = sum(doc_counts.values())
        print(f"\n=== {lab}: {total} spans in {len(doc_counts)} documents ===")
        print("Top documents:", doc_counts.most_common(8))
        print("Largest document share:", round(max(doc_counts.values(), default=0)/max(total,1), 3))
        print("Top annotated surfaces:", surface_counts.most_common(12))
        print("Examples (context-before | annotated-span | context-after):")
        for ex in examples:
            print(f"doc={ex[0]} [{ex[1]}:{ex[2]}] ...{ex[3]!r} [[{ex[4]!r}]] {ex[5]!r}")

print("TRAIN LABEL COUNTS")
print(Counter(str(e[2]) for ents in train_gold.values() for e in ents if len(e) >= 3))
label_audit(train_texts, train_gold)
label_audit(val_texts, val_gold)

In [ ]:
# 9. Token-level diagnostic: what labels compete at gold RESP/WIT positions?
from collections import defaultdict

def gold_token_probability_audit(model, texts, gold, labels=("RESP", "WIT")):
    summaries = defaultdict(lambda: {"probs": [], "top": Counter(), "B_top": 0, "I_top": 0, "B_n": 0, "I_n": 0})
    for doc_id, text in tqdm(texts.items(), desc="Gold-token audit", leave=False):
        offsets, probs = diag_doc_probs(model, text)
        if not len(offsets): continue
        top_ids = probs.argmax(-1).tolist()
        for ann in gold[str(doc_id)]:
            if len(ann) < 3: continue
            s, e, lab = int(ann[0]), int(ann[1]), str(ann[2])
            if lab not in labels: continue
            ix = [i for i,(a,b) in enumerate(offsets) if b > a and a < e and b > s]
            for j, i in enumerate(ix):
                target_tag = ("B-" if j == 0 else "I-") + lab
                tid = label2id[target_tag]
                summaries[lab]["probs"].append(float(probs[i, tid]))
                summaries[lab]["top"][BIO_LABELS[top_ids[i]]] += 1
                key = "B" if j == 0 else "I"
                summaries[lab][key+"_n"] += 1
                summaries[lab][key+"_top"] += int(top_ids[i] == tid)
    for lab in labels:
        d = summaries[lab]
        print(f"\n{lab}: mean gold target-tag probability={np.mean(d['probs']) if d['probs'] else float('nan'):.4f}")
        print(f"B top1={d['B_top']}/{d['B_n']} | I top1={d['I_top']}/{d['I_n']}")
        print("Competing top labels:", d["top"].most_common(10))

gold_token_probability_audit(champion_model, train_texts, train_gold)
gold_token_probability_audit(champion_model, val_texts, val_gold)

In [ ]:
# 10. RESP/WIT logit-bias diagnostic on validation only
# Does not train, save, or alter model weights.
bias_values = [0.0, 0.5, 1.0, 1.5, 2.0, 3.0]
bias_results = {}
for bias in bias_values:
    candidate = {}
    for doc_id, text in tqdm(val_texts.items(), desc=f"Bias {bias}", leave=False):
        offsets, probs = diag_doc_probs(champion_model, text)
        if not len(offsets):
            candidate[doc_id] = []; continue
        logits = torch.log(probs.clamp_min(1e-12))
        logits = logits.clone()
        for lab in ("RESP", "WIT"):
            logits[:, label2id["B-"+lab]] += bias
            logits[:, label2id["I-"+lab]] += bias
        adjusted_probs = torch.softmax(logits, dim=-1)
        pred_ids = bio_viterbi_decode(torch.log(adjusted_probs.clamp_min(1e-12)))
        spans = decode_spans(text, offsets, pred_ids)
        candidate[doc_id] = spans
    metrics = calculate_exact_span_metrics(val_gold, candidate)
    bias_results[bias] = (candidate, metrics)
    print({
        "bias": bias, "macro_f1": round(metrics["MACRO_F1"], 6),
        "RESP_f1": round(metrics["RESP"]["f1"], 4),
        "WIT_f1": round(metrics["WIT"]["f1"], 4),
        "RESP_pred": metrics["RESP"]["pred"], "WIT_pred": metrics["WIT"]["pred"],
        "total_pred": sum(len(v) for v in candidate.values())
    })
print("\nThis is a diagnostic; bias candidates must not replace the champion automatically.")

In [ ]:
# 11. Confidence filtering diagnostic for bias=2 (validation only)
# This isolates whether high-confidence additions can improve the threshold-only baseline.
bias2_candidate, bias2_metrics = bias_results[2.0]
# Recreate adjusted probabilities and span confidence from predicted BIO tags.
bias2_scored = {}
for doc_id, text in tqdm(val_texts.items(), desc="Bias-2 span confidence", leave=False):
    offsets, probs = diag_doc_probs(champion_model, text)
    if not len(offsets):
        bias2_scored[doc_id] = []; continue
    logits = torch.log(probs.clamp_min(1e-12)).clone()
    for lab in ("RESP", "WIT"):
        logits[:, label2id["B-"+lab]] += 2.0
        logits[:, label2id["I-"+lab]] += 2.0
    adj = torch.softmax(logits, dim=-1)
    ids = bio_viterbi_decode(torch.log(adj.clamp_min(1e-12)))
    tag_conf = adj[torch.arange(len(ids)), torch.tensor(ids, dtype=torch.long)].tolist()
    spans = decode_spans(text, offsets, ids)
    scored = []
    for s,e,lab in spans:
        if lab not in ("RESP", "WIT"): continue
        ix = [i for i,(a,b) in enumerate(offsets) if b > a and a < e and b > s]
        conf = float(np.mean([tag_conf[i] for i in ix])) if ix else 0.0
        scored.append((s,e,lab,conf))
    bias2_scored[doc_id] = scored

for thr in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    cand = {d: list(threshold_val.get(d, [])) for d in val_texts}
    for d, spans in bias2_scored.items():
        for s,e,lab,conf in spans:
            if conf < thr: continue
            if any(s < pe and e > ps for ps,pe,_ in cand[d]):
                continue
            cand[d].append((s,e,lab))
    m = calculate_exact_span_metrics(val_gold, cand)
    print({"threshold":thr, "macro_f1":round(m["MACRO_F1"],6),
           "RESP_f1":round(m["RESP"]["f1"],4), "WIT_f1":round(m["WIT"]["f1"],4),
           "RESP_pred":m["RESP"]["pred"], "WIT_pred":m["WIT"]["pred"],
           "total_pred":sum(len(v) for v in cand.values())})
print("\nNote: this hybrid is compared against threshold-only baseline, not the full rule-based champion.")

In [ ]:
# 12. Role-cue audit (not a prediction rule)
# Counts whether legal role language appears near gold RESP/WIT mentions.
ROLE_CUES = {
    "RESP": re.compile(r"\b(respondent|defendant|accused|opposite party|non-applicant)\b", re.I),
    "WIT": re.compile(r"\b(witness|deposed|testified|statement of|examined as a witness)\b", re.I),
    "PETITIONER": re.compile(r"\b(petitioner|applicant|appellant|complainant|plaintiff)\b", re.I),
}
def cue_context_audit(texts, gold, window=300):
    for lab in ("RESP", "WIT"):
        total = near_any = near_same = 0
        examples = []
        for d, ents in gold.items():
            text = texts[d]
            for ent in ents:
                if len(ent) < 3 or str(ent[2]) != lab: continue
                s,e = int(ent[0]),int(ent[1]); total += 1
                ctx = text[max(0,s-window):min(len(text),e+window)]
                any_cue = any(rx.search(ctx) for rx in ROLE_CUES.values())
                same_cue = bool(ROLE_CUES[lab].search(ctx))
                near_any += int(any_cue); near_same += int(same_cue)
                if len(examples) < 5:
                    examples.append((d, text[max(0,s-100):s], text[s:e], text[e:min(len(text),e+100)], any_cue, same_cue))
        print(f"\n{lab}: gold={total}, any role cue within ±{window} chars={near_any} ({near_any/max(total,1):.1%}), same-label cue={near_same} ({near_same/max(total,1):.1%})")
        for x in examples:
            print(f"doc={x[0]} ...{x[1]!r} [[{x[2]!r}]] {x[3]!r} | anycue={x[4]} samecue={x[5]}")
cue_context_audit(train_texts, train_gold)
cue_context_audit(val_texts, val_gold)

In [ ]:
# 13. Checkpoint decision report
# We use the champion recovery checkpoint for inference. Other checkpoints are candidates only
# if their architecture, label order, and validation score can be established explicitly.
print("Current NER checkpoint:", CHAMPION_PATH)
print("Phase-1 encoder checkpoint:", PHASE1_PATH)
print("Champion validation threshold-only macro-F1:", round(threshold_metrics["MACRO_F1"], 6))
print("Best established full champion+rules reference from prior notebook: 0.299526")
print("No checkpoint was modified or saved.")
print("\nTraining is intentionally not run automatically: prior targeted fine-tuning and frozen-head experiments regressed.")
print("Next decision should be based on the annotation examples, role-cue coverage, and validation error analysis above.")